# Replication Study — Data Ingestion (adult, churn_modelling, covertype, cardio)

Runs the 4 new loaders against the datasets used in Yan et al. (VAE-GAN,
KBS 2025), producing `train.csv` / `test.csv` / `meta.json` under
`data/processed/{dataset}/` — same shape your existing evaluation
notebooks already read.

**Before running:**
- `adult`: download UCI Adult (`adult.data` + `adult.test`), concatenate
  into one file, strip the `adult.test` header comment line, save as
  `data/raw/adult_combined.csv`.
- `churn_modelling`: download from Kaggle
  (`shrutimechlearn/churn-modelling`), save as `data/raw/churn_modelling.csv`.
- `covertype`: no download needed if `use_sklearn_fetch: true` in config
  (default) — `sklearn.datasets.fetch_covtype()` handles it and caches
  locally. Needs internet access on first run.
- `cardio`: download from Kaggle (`sulianova/cardiovascular-disease-dataset`),
  save as `data/raw/cardio_train.csv` (semicolon-delimited, keep as-is).

Paste the entries from `config_additions.yaml` into your real `config.yaml`
under `datasets:` before running this.

In [1]:
import os
os.chdir('..')

import yaml
with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.ingestion.adult_loader import load_and_process as load_adult
from src.ingestion.churn_modelling_loader import load_and_process as load_churn
from src.ingestion.covertype_loader import load_and_process as load_covertype
from src.ingestion.cardio_loader import load_and_process as load_cardio

REPLICATION_DATASETS = ['adult', 'churn_modelling', 'covertype', 'cardio']
missing = [d for d in REPLICATION_DATASETS if d not in config['datasets']]
if missing:
    raise KeyError(
        f'{missing} not found in config.yaml["datasets"] — paste the entries '
        f'from config_additions.yaml into your config.yaml first.'
    )
print('Config entries found for all 4 replication datasets.')

Config entries found for all 4 replication datasets.


In [2]:
loaders = {
    'adult': load_adult,
    'churn_modelling': load_churn,
    'covertype': load_covertype,
    'cardio': load_cardio,
}

results = {}
for name, loader_fn in loaders.items():
    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    try:
        loader_fn(config)
        results[name] = 'OK'
    except FileNotFoundError as e:
        print(f'  SKIPPED — raw file not found: {e}')
        results[name] = 'SKIPPED (raw file missing)'
    except Exception as e:
        print(f'  ERROR: {e}')
        results[name] = f'ERROR: {e}'

print('\n\nSummary:')
for name, status in results.items():
    print(f'  {name}: {status}')


  adult

  churn_modelling

  covertype

  cardio


Summary:
  adult: OK
  churn_modelling: OK
  covertype: OK
  cardio: OK


## Sanity check each processed dataset

Loads each `train.csv`/`meta.json` back and prints shape, target
distribution, and categorical column count — quick visual check before
handing these to the generator pipeline.

In [3]:
import json
import pandas as pd
from pathlib import Path

for name in REPLICATION_DATASETS:
    processed_dir = Path(config['datasets'][name]['processed_dir'])
    if not (processed_dir / 'train.csv').exists():
        continue
    train = pd.read_csv(processed_dir / 'train.csv')
    test = pd.read_csv(processed_dir / 'test.csv')
    meta = json.load(open(processed_dir / 'meta.json'))
    target = meta['target_col']
    n_cat = sum(1 for c in meta['columns'].values() if c['type'] == 'categorical')

    print(f'\n{name}:')
    print(f'  train={train.shape}, test={test.shape}, target={target}, categorical_cols={n_cat}')
    print(f'  target distribution:\n{train[target].value_counts(normalize=True).round(3).to_string()}')


adult:
  train=(36177, 14), test=(9045, 14), target=income, categorical_cols=9
  target distribution:
income
<=50K    0.752
>50K     0.248

churn_modelling:
  train=(8000, 11), test=(2000, 11), target=Exited, categorical_cols=5
  target distribution:
Exited
0    0.796
1    0.204

covertype:
  train=(24000, 13), test=(6000, 13), target=Cover_Type, categorical_cols=3
  target distribution:
Cover_Type
2    0.485
1    0.369
3    0.061
7    0.035
6    0.030
5    0.016
4    0.005

cardio:
  train=(24000, 12), test=(6000, 12), target=cardio, categorical_cols=7
  target distribution:
cardio
1    0.503
0    0.497
